In [28]:
from pathlib import Path

from ai_tools.data_extractor.tools import extract_funcs as ef

import re

In [29]:
save_data = Path('rag_data')

RAW ICT283 DATA

In [30]:
ict_data_pth = Path().cwd() / 'ai_tools' / 'data_extractor' / 'data' / 'raw'

txt_files = ef.search_txt_files(ict_data_pth)

In [31]:
def preamble_extract(pth):
    
    with open(pth, 'r', encoding='utf-8') as f:
        
        lines = ""
        temp = f.readline()
        
        while not re.search(r"^([Qq].?)$", temp.strip()):
            lines += temp
            temp = f.readline()
            
        lines = re.sub(r"^(-+)$", "", lines,flags=re.MULTILINE)
        lines = lines.strip()
        lines += '\n'
        
    return lines 

In [32]:
def combine_and_refactor_files(list : list[str], save_pth : str):
    
    to_write = ""
    
    for pth in list:
        
        preamble = re.sub(r'\n+',' ', preamble_extract(pth))
        preamble = preamble if len(preamble) > 0 else "new topic"
        
        to_write += "Context: " + preamble + '\n'
        
        for pairs in ef.extract_qna_pairs([pth]):
                
            refactored_pair = f'Question: {pairs['Q'].strip()} Answer: {pairs['A'].strip()}'
            refactored_pair = re.sub(r'\n+',' ', refactored_pair)
            refactored_pair = refactored_pair.strip() + '\n'
            to_write += refactored_pair
            
    with open(save_pth, 'w') as f:
        
        f.write(to_write)
        
        
    return to_write

combine_and_refactor_files(txt_files, save_data / 'ICT283_cleaned_raws.txt')

'Context: Assignment Moderation (Assignments 1 and 2) \tBe aware that your assignment 1 and 2 marks are not final until your exam performance validates your ability. We reserve the right to revoke your assignment marks (0 marks) if your exam answers indicate that your ability is lower than what is indicated by your performance in the assignments. When this happens, you may be required to sit an invigilated assignment that is handwritten on paper. You will need to pass this invigilated hand-written assignment for your original assignment marks to be re-instated.  There can be multiple questions related to the same topic. What that means that questions about similar things are asked looking for emphasis on different aspects. So use the find function to locate something. Download this file first. The Questions and Answers (QandA) list shown below is in reverse chronological order. It is very important that you are aware of the contents of this file and follow the advice shown here.  The a

STACK OVERFLOW DATA

In [44]:
stack_oflow_path = Path().cwd() / 'ai_tools' / 'data_preprocessing' / 'processed_data' / 'train_ready.parquet'
print('exists') if stack_oflow_path.exists() and stack_oflow_path.is_file() else print('does not exist')

exists


In [46]:
import pandas as pd

oflow_df = pd.read_parquet(stack_oflow_path)
oflow_df.head()

,topic,question_title,question_body,answer_body,label
0,ABSTRACT CLASSES,Interfaces vs Types in TypeScript,What is the difference between these statement...,Update March 2021: The newer TypeScript Handbo...,1
1,ABSTRACT CLASSES,Can an abstract class have a constructor?,Can an abstract class have a constructor? If s...,"Yes, an abstract class can have a constructor....",1
2,ABSTRACT CLASSES,How to determine if a type implements an inter...,Does reflection in C# offer a way to determine...,You have a few choices: typeof(IMyInterface).I...,1
3,ABSTRACT CLASSES,C# Interfaces. Implicit implementation versus ...,What are the differences in implementing inter...,Implicit is when you define your interface via...,1
4,ABSTRACT CLASSES,Why can't static methods be abstract in Java?,The question is in Java why can't I define an ...,The abstract annotation to a method indicates ...,1


In [58]:
def make_single_line(string_in):
    
    string_out = re.sub(r'\n+', ' ', string_in)
    string_out = string_out.strip()
    
    return string_out

def sample_of_oflow_df(df, n = 100):
    
    new_df = oflow_df.groupby(by=['topic']).sample(n=n).drop(columns=['label'])
    singled_df = new_df.map(make_single_line)
    
    return singled_df

def  create_oflow_txt(df : pd.DataFrame, save_pth : str):
    
    to_write = ""
    for _, row in df.iterrows():
        
        topic : str = row['topic']
        question_title : str = row['question_title']
        question_body : str = row['question_body']
        answer_body : str = row['answer_body']
        
        row_formatted = f'Topic: {topic.lower()} Question: {question_title} {question_body} Answer: {answer_body}'
        row_formatted += '\n'
        
        to_write += row_formatted
        
    with open(str(save_pth), 'w') as f:
        
        f.write(to_write)

new_oflow_df = sample_of_oflow_df(oflow_df)
create_oflow_txt(new_oflow_df, save_data / 'stack_overflow_data.txt')